## Home Task

- 1. Replicate Simple recommender implementation
- 2. (optional) Replicate the the content based recommender implementation

### 1. Replicate Simple recommender implementation

- Offer generalized recommendations to every user, based on movie popularity and/or genre. The basic idea behind this system is that movies that are more popular and critically acclaimed will have a higher probability of being liked by the average audience

In [221]:
# Import library
import pandas as pd

In [222]:
# Load Movies Metadata
metadata = pd.read_csv('Data/movies_metadata.csv', encoding='utf-8-sig')

# Print the first three rows
metadata.head(3)

C:\Users\Admin\AppData\Local\Temp\ipykernel_7984\782477885.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv('Data/movies_metadata.csv', encoding='utf-8-sig')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [223]:
# Calculate mean of vote average column
C = metadata['vote_average'].mean()
print(C)

5.618207215134184


In [224]:
# Calculate the minimum number of votes required to be in the chart, m
m = metadata['vote_count'].quantile(0.90)
print(m)

160.0


In [225]:
# Filter out all qualified movies into a new DataFrame
q_movies = metadata.copy().loc[metadata['vote_count'] >= m]
q_movies.shape

(4555, 24)

In [226]:
# Function that computes the weighted rating of each movie
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    # Calculation based on the IMDB formula
    return (v/(v+m) * R) + (m/(m+v) * C)


In [227]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

#Sort movies based on score calculated above
q_movies = q_movies.sort_values('score', ascending=False)

Output the top 15 movies 

In [228]:
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


### (optional) Replicate the the content based recommender implementation

- This system uses item metadata, such as genre, director, description, actors, etc. for movies, to make these recommendations. The general idea if a person likes a particular item, he or she will also like an item that is similar to it. And to recommend that, it will make use of the user's past item metadata

In [229]:
#Import library 
import numpy as np
import pandas as pd

from scipy.sparse import lil_matrix 
from sklearn.metrics.pairwise import cosine_similarity

#TfIdfVectorizer from scikit-learn 
from sklearn.feature_extraction.text import TfidfVectorizer 


In [230]:
#Print plot overviews of the first 5 movies.
metadata['overview'].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: object

In [231]:
#Define a TF-IDF Vectorizer Object. Remove all english stop words such as 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

#Replace NaN with an empty string
metadata['overview'] = metadata['overview'].fillna('')

#Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(metadata['overview'])

#Output the shape of tfidf_matrix
tfidf_matrix.shape

(45466, 75827)

In [232]:
#Array mapping from feature integer indices to feature name.
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [233]:
def compute_sparse_similarity(tfidf_matrix, threshold, chunk_size):
    """
    Calculates similarity and stores the result as a sparse array (only values > threshold), using a vectorized approach
    """ 

    n_samples = tfidf_matrix.shape[0]
    sparse_sim  = lil_matrix((n_samples, n_samples), dtype = np.float32)
    
    # processing in batches
    for i in range(0, n_samples, chunk_size):
        end = min(i + chunk_size, n_samples)
        print(f'Processing batch {i}-{end} with {n_samples} ...')

        # Let's grab a batch of movies
        batch = tfidf_matrix[i:end]

        # For entire batch calculates similarity 
        batch_sim  = cosine_similarity(batch, tfidf_matrix).astype(np.float32)
        
        # Saving the result
        for row_in_batch, global_idx in enumerate(range(i, end)):
            sim_row = batch_sim[row_in_batch]

            # Find the indices where the similarity is greater than the threshold
            mask = sim_row > threshold
            mask[global_idx] = False
            if mask.sum() > 0:
                cols = np.where(mask)[0]
                vals = sim_row[mask]
                sparse_sim[global_idx, cols] = vals
        del batch_sim
    
    return sparse_sim.tocsr()


In [ ]:
# Convert the result to a sparse matrix
sparse_sim = compute_sparse_similarity(tfidf_matrix, threshold=0.1, chunk_size= 1000)

print(f"\nSparse matrix similarity: {sparse_sim.shape}")
print(f"Non-zero connections: {sparse_sim.nnz}")
print(f"Average number of links per film: {sparse_sim.nnz / tfidf_matrix.shape[0]:.1f}")
print(f"Memory Size: {sparse_sim.data.nbytes / 1024 / 1024:.2f} MB")


Processing batch 0-1000 with 45466 ...
Processing batch 1000-2000 with 45466 ...
Processing batch 2000-3000 with 45466 ...
Processing batch 3000-4000 with 45466 ...
Processing batch 4000-5000 with 45466 ...
Processing batch 5000-6000 with 45466 ...
Processing batch 6000-7000 with 45466 ...
Processing batch 7000-8000 with 45466 ...
Processing batch 8000-9000 with 45466 ...
Processing batch 9000-10000 with 45466 ...
Processing batch 10000-11000 with 45466 ...
Processing batch 11000-12000 with 45466 ...
Processing batch 12000-13000 with 45466 ...
Processing batch 13000-14000 with 45466 ...
Processing batch 14000-15000 with 45466 ...
Processing batch 15000-16000 with 45466 ...
Processing batch 16000-17000 with 45466 ...
Processing batch 17000-18000 with 45466 ...
Processing batch 18000-19000 with 45466 ...
Processing batch 19000-20000 with 45466 ...
Processing batch 20000-21000 with 45466 ...
Processing batch 21000-22000 with 45466 ...
Processing batch 22000-23000 with 45466 ...
Processing

In [ ]:
def get_recommendations_sparse(title, metadata, sparse_sim, n=10):
    """
    Recommendations for using the sparse similarity matrix
    """

    # Creating a dictionary from title to index
    indices = pd.Series(metadata.index, index=metadata['title'])
    
    if title not in indices:
        return f"Movies '{title}' not founded"
    
    idx = indices[title]
    
    # Retrieving a row from a sparse matrix 
    sim_row = sparse_sim[idx]
    
    # Check if there are any recommendations
    if sim_row.nnz == 0:
        return f"No recommendations for this movie '{title}'"
    
    # Retrieving the IDs and ratings of similar movies
    similar_indices = sim_row.indices
    similar_scores = sim_row.data
    
    # Sort in descending order
    sorted_order = np.argsort(similar_scores)[::-1]
    sorted_indices = similar_indices[sorted_order]
    sorted_scores = similar_scores[sorted_order]
    
    # Take the top N 
    top_indices = sorted_indices[:n]
    top_scores = sorted_scores[:n]
    
    # Get the result
    recommendations = pd.DataFrame({
        'title': metadata.iloc[top_indices]['title'].values,
        'similarity': top_scores
    }) 

    return recommendations


Output 5 movies similar to *"The Dark Knight Rises"*

In [ ]:
get_rec = get_recommendations_sparse('The Dark Knight Rises', metadata, sparse_sim, n=5)
get_rec
    

,title,similarity
0,The Dark Knight,0.326639
1,Batman Forever,0.316211
2,Batman Returns,0.296139
3,Batman: Under the Red Hood,0.287983
4,Batman,0.270730
